# Конфиг и проверка ГПУ

In [1]:
from pathlib import Path
import json
import torch
import os

# Вся HF-экосистема (hub, datasets и т.д.) — на диск S:
os.environ["HF_HOME"] = r"S:\hw\aspa\.cache\huggingface"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ROOT = Path(r"S:\hw\aspa")
MODEL_ID = "Qwen/Qwen3.5-2B"

ADAPTER_NAME = "lora_full_bf16_6k"
OUTPUT_DIR = ROOT / "output" / ADAPTER_NAME
SEED_DIR = ROOT / "llm_fc_hypernym_prediction/output/dataset/seed_42"
TRAIN_LIMIT = None
EVAL_LIMIT = None
MAX_LEN = 6000

if OUTPUT_DIR.exists():
    raise FileExistsError(
        f"Адаптер уже существует: {OUTPUT_DIR} — смени ADAPTER_NAME"
    )

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0))
print("OUTPUT_DIR:", OUTPUT_DIR)
print("SEED_DIR:", SEED_DIR)

torch: 2.11.0+cu128
cuda: True NVIDIA GeForce RTX 5080
OUTPUT_DIR: S:\hw\aspa\output\lora_full_bf16_6k
SEED_DIR: S:\hw\aspa\llm_fc_hypernym_prediction\output\dataset\seed_42


# Загрузить и посмотреть данные

In [2]:
def load_jsonl(path: Path) -> list[dict]:
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


train_records_raw = load_jsonl(SEED_DIR / "train.jsonl")
eval_records_raw = load_jsonl(SEED_DIR / "eval.jsonl")
print("train.jsonl:", len(train_records_raw), "eval.jsonl:", len(eval_records_raw))
if train_records_raw:
    print("roles:", [m["role"] for m in train_records_raw[0]["messages"]])
    print("final:", train_records_raw[0]["messages"][-1]["content"][:120])

train.jsonl: 2504 eval.jsonl: 355
roles: ['system', 'user', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant']
final: hyponym of 3240-N (спортивный объект)


# Загрузка туллзов

In [3]:
import sys
sys.path.insert(0, str(ROOT / "llm_fc_hypernym_prediction"))
from utils.tools import hyponym_only

TOOLS = hyponym_only
TOOLS[0]  # посмотреть схему get_hyponyms

{'type': 'function',
 'function': {'name': 'get_hyponyms',
  'description': 'Navigate the RuWordNet taxonomy by retrieving hyponyms (more specific concepts) of a given synset. Returns formatted markdown with: synset name, ID, associated words, and list of child hyponyms. When node_id is null, returns all root nodes (top-level concepts). Use this to explore the taxonomy tree level by level.',
  'parameters': {'type': 'object',
   'properties': {'node_id': {'type': ['string', 'null'],
     'description': "The synset ID to get hyponyms for. Use 'null' or null to retrieve all root nodes (top-level concepts in the taxonomy). Use specific synset ID like '123456-N' to get its children."}},
   'required': ['node_id']}}}

# Загрузка tokenizer + проверка chat template

In [4]:
import copy
import json

def normalize_messages_for_template(messages):
    """Qwen3.5 chat template ждёт arguments как dict, не JSON-строку."""
    msgs = copy.deepcopy(messages)
    for msg in msgs:
        if msg.get("role") != "assistant":
            continue
        for tc in msg.get("tool_calls") or []:
            fn = tc.get("function", tc)
            args = fn.get("arguments")
            if isinstance(args, str):
                fn["arguments"] = json.loads(args)
    return msgs

In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
sample = normalize_messages_for_template(train_records_raw[0]["messages"][:3])
text = tokenizer.apply_chat_template(
    sample,
    tools=TOOLS,
    tokenize=False,
    add_generation_prompt=False,
    enable_thinking=False,
)
print(text[:2000])

<|im_start|>system
# Tools

You have access to the following functions:

<tools>
{"type": "function", "function": {"name": "get_hyponyms", "description": "Navigate the RuWordNet taxonomy by retrieving hyponyms (more specific concepts) of a given synset. Returns formatted markdown with: synset name, ID, associated words, and list of child hyponyms. When node_id is null, returns all root nodes (top-level concepts). Use this to explore the taxonomy tree level by level.", "parameters": {"type": "object", "properties": {"node_id": {"type": ["string", "null"], "description": "The synset ID to get hyponyms for. Use 'null' or null to retrieve all root nodes (top-level concepts in the taxonomy). Use specific synset ID like '123456-N' to get its children."}}, "required": ["node_id"]}}}
</tools>

If you choose to call a function ONLY reply in the following format with NO suffix:

<tool_call>
<function=example_function_name>
<parameter=example_parameter_1>
value_1
</parameter>
<parameter=example_p

# Загрузка модели

### Загрузка Qwen3.5-2B (bf16, без квантизации)

In [6]:
import gc
import torch
from transformers import AutoModelForCausalLM

gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


[ERROR] `loss` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in s:\hw\aspa\.ml_venv\Lib\site-packages\transformers\models\qwen3_5\modeling_qwen3_5.py.
[ERROR] `logits` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in s:\hw\aspa\.ml_venv\Lib\site-packages\transformers\models\qwen3_5\modeling_qwen3_5.py.


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
s:\hw\aspa\.ml_venv\Lib\site-packages\accelerate\utils\modeling.py:817: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  _ = torch.tensor([0], device=i)


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

# LoRA + SFT

### Подготовка датасета

In [7]:
from datasets import Dataset

def count_tokens(messages: list[dict]) -> int:
    normalized = normalize_messages_for_template(messages)
    result = tokenizer.apply_chat_template(
        normalized,
        tools=TOOLS,
        tokenize=True,
        return_dict=True,
        enable_thinking=False,
    )
    return len(result["input_ids"])


def filter_by_length(records: list[dict], limit: int | None) -> tuple[list[dict], int]:
    kept: list[dict] = []
    skipped = 0
    for record in records:
        n = count_tokens(record["messages"])
        if n < MAX_LEN:
            kept.append(record)
        else:
            skipped += 1
        if limit is not None and len(kept) >= limit:
            break
    return kept, skipped


train_filtered, train_skipped = filter_by_length(train_records_raw, TRAIN_LIMIT)
eval_filtered, eval_skipped = filter_by_length(eval_records_raw, EVAL_LIMIT)

print(
    f"train: kept={len(train_filtered)}, skipped_by_len={train_skipped}, "
    f"limit={TRAIN_LIMIT}, max_len<{MAX_LEN}"
)
print(
    f"eval: kept={len(eval_filtered)}, skipped_by_len={eval_skipped}, "
    f"limit={EVAL_LIMIT}, max_len<{MAX_LEN}"
)

records_for_train = [
    {
        "messages": normalize_messages_for_template(r["messages"]),
        "tools": TOOLS,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    for r in train_filtered
]
records_for_eval = [
    {
        "messages": normalize_messages_for_template(r["messages"]),
        "tools": TOOLS,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    for r in eval_filtered
]

ds_train = Dataset.from_list(records_for_train)
ds_eval = Dataset.from_list(records_for_eval)
print("ds_train:", len(ds_train), "ds_eval:", len(ds_eval))

train: kept=938, skipped_by_len=1566, limit=None, max_len<6000
eval: kept=124, skipped_by_len=231, limit=None, max_len<6000
ds_train: 938 ds_eval: 124


### Проверка длин до обучения

In [8]:
lengths_train = [count_tokens(r["messages"]) for r in train_filtered]
lengths_eval = [count_tokens(r["messages"]) for r in eval_filtered]
if lengths_train:
    print(
        f"train tokens: min={min(lengths_train)}, max={max(lengths_train)}, "
        f"avg={sum(lengths_train)/len(lengths_train):.0f}"
    )
if lengths_eval:
    print(
        f"eval tokens: min={min(lengths_eval)}, max={max(lengths_eval)}, "
        f"avg={sum(lengths_eval)/len(lengths_eval):.0f}"
    )

train tokens: min=1627, max=5994, avg=4584
eval tokens: min=1753, max=5950, avg=4667


### Запуск LoRA (bf16, без QLoRA)

In [9]:
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()

gc.collect()
torch.cuda.empty_cache()

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_steps=50,
    max_length=MAX_LEN,
    assistant_only_loss=True,
    gradient_checkpointing=True,
    save_total_limit=3,
    loss_type="chunked_nll",
    report_to="none",

    eval_strategy="steps",
    eval_steps=50,
    eval_on_start=True,
    per_device_eval_batch_size=1,

    load_best_model_at_end=True,
    metric_for_best_model="loss",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_eval,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(str(OUTPUT_DIR))

W0630 11:03:15.035000 28744 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


trainable params: 737,280 || all params: 1,882,562,368 || trainable%: 0.0392


Tokenizing train dataset:   0%|          | 0/938 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/124 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
0,No log,0.255278,0.154920,0.000000,0.958525
50,0.047182,0.047774,0.044466,918556.000000,0.980705
100,0.038185,0.045806,0.032768,1836084.000000,0.982871
150,0.036092,0.041141,0.035485,2744476.000000,0.982649
200,0.035747,0.037637,0.033918,3652762.000000,0.984401
235,0.032503,0.037052,0.035097,4300126.000000,0.984636


In [10]:
# import torch
# torch.cuda.reset_peak_memory_stats()
# for i, batch in enumerate(trainer.get_train_dataloader()):
#     trainer.training_step(model, batch)
#     seq_len = batch["input_ids"].shape[1]
#     peak = torch.cuda.max_memory_allocated() / 1e9
#     print(f"step {i}: seq={seq_len}, peak={peak:.2f} GB")
#     torch.cuda.reset_peak_memory_stats()